# 02 — Mamba Architecture Lab

Prototype and explore the **Mamba** architecture using the building blocks in
`src/core/`.  This is where you iterate before promoting to `src/`.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from src.models.mamba.config import MambaConfig
from src.models.mamba.model import MambaSLM
from src.core.generation import generate
from src.utils.training import count_params

In [2]:
# ── Build a tiny model for fast iteration ────────────────────────────────
tiny_cfg = MambaConfig(
    vocab_size=50257, block_size=32,
    n_layer=4, d_model=64,
    d_state=16, d_conv=4, expand=2,
    dropout=0.0,
)
model = MambaSLM(tiny_cfg)
print(f'Parameters: {count_params(model):,}')

Parameters: 3,344,256


In [3]:
# ── Verify forward pass shapes ────────────────────────────────────────────
B, T = 2, tiny_cfg.block_size
idx = torch.randint(0, tiny_cfg.vocab_size, (B, T))
targets = torch.randint(0, tiny_cfg.vocab_size, (B, T))

logits, loss = model(idx, targets)
print(f'logits: {logits.shape}  |  loss: {loss.item():.4f}')

logits: torch.Size([2, 32, 50257])  |  loss: 10.8377


In [4]:
# ── Test generation ───────────────────────────────────────────────────────
prompt = torch.randint(0, tiny_cfg.vocab_size, (1, 4))
output = generate(model, prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f'Input tokens:  {prompt[0].tolist()}')
print(f'Output tokens: {output[0].tolist()}')

Input tokens:  [12430, 7688, 4100, 48620]
Output tokens: [12430, 7688, 4100, 48620, 16563, 7379, 4449, 40414, 8709, 42957, 19257, 12894, 39877, 3819]


In [5]:
# ── Parameter count comparison across model sizes ────────────────────────
configs = {
    'mamba_tiny':   MambaConfig(vocab_size=50257, block_size=16,  n_layer=4,  d_model=64,  d_state=16, d_conv=4, expand=2),
    'mamba_small':  MambaConfig(vocab_size=50257, block_size=128, n_layer=12, d_model=384, d_state=16, d_conv=4, expand=2),
    'mamba_medium': MambaConfig(vocab_size=50257, block_size=256, n_layer=24, d_model=768, d_state=16, d_conv=4, expand=2),
}
for name, cfg in configs.items():
    n = count_params(MambaSLM(cfg))
    print(f'{name:14s}: {n:>12,} params  ({n/1e6:.1f}M)')

mamba_tiny    :    3,344,256 params  (3.3M)
mamba_small   :   30,445,824 params  (30.4M)
mamba_medium  :  125,652,480 params  (125.7M)
